In [1]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
import requests
import gspread,os,pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from sodapy import Socrata
import platform
platform  =  platform.platform()
if platform == "Linux-2.6.32-754.35.1.el6.x86_64-x86_64-with-centos-6.10-Final":
    bic_etl_home="/usr/local/cim/bic_etl"
else:
    bic_etl_home = os.getenv('bic_etl_home')

In [5]:
scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

#creds = ServiceAccountCredentials.from_json_keyfile_name('../../scripts/client_secret.json',
#    scope)
creds = ServiceAccountCredentials.from_json_keyfile_name(os.path.join(bic_etl_home, 'general', 'scripts','client_secret.json'),scope)

client = gspread.authorize(creds)
tracker = client.open('BIC Dataset Tracker').worksheet(
'PublishedData')
metadata_df = pd.DataFrame(tracker.get_all_records(head=2))


In [6]:
metadata_df

,Dataset Title,Short Description,Category,Keywords,Type,License Type,Data Provider,Data Provided by,Source Link,State Steward,...,Business Contact Information - Name,Business Contact Information - Position,Business Contact Information - Phone Number,Business Contact Information - Email,Technical Contact Information - Name,Technical Contact Information - Position,Technical Contact Information - Phone Number,Technical Contact Information - Email,Name of Person Providing Data,Email of Person Providing Data
0,Bill Information and Position with Income of L...,Information for each lobbyist and their associ...,Lobbyist,"bic, gocodecolorado, colorado, sos, secretary ...",Bills,Public Domain,CDOS,CDOS - Colorado Department of State,,CDOS,...,,BIC Program Manager,,,Angela Lawson,Lobbyist Registration Program Manager,"303-894-2200, ext. 6304",a.lawson@sos.state.co.us,Sebbie Coleman,sebbie.coleman@sos.state.co.us
1,Expenses for Lobbyists in Colorado,Registered lobbyist expenses for the State of ...,Lobbyist,"bic, gocodecolorado, colorado, sos, secretary ...",Bills,Public Domain,CDOS,CDOS - Colorado Department of State,,CDOS,...,,BIC Program Manager,,,Angela Lawson,Lobbyist Registration Program Manager,"303-894-2200, ext. 6304",a.lawson@sos.state.co.us,Sebbie Coleman,sebbie.coleman@sos.state.co.us
2,State Lobbyist Bills in Colorado,Bills that state liasons (lobbyists) monitor. ...,Legislative,"bic, gocodecolorado, colorado, sos, secretary ...",Bills,Public Domain,CDOS,,,,...,Myra Rooney,Colorado Lobby Program Lead Trainer,303.894.2200 (x 6304),Myra.Rooney@coloradosos.gov,John Coniff,,,john.coniff@coloradosos.gov,John Coniff,john.coniff@coloradosos.gov
3,Biomass Residue in Colorado,Biopower potential estimated based on availabl...,Energy,"bic, gocodecolorado, colorado, nrel, national ...",Biomass,Public Domain,NREL,NREL - National Renewable Energy Laboratory,https://maps.nrel.gov/re-atlas/?aL=0&bL=clight...,,...,,,,,,,,,,
4,Municipal Annexations in Colorado,"Annexation data for Colorado Municipalities, f...",Local Aggregation,"bic, gocodecolorado, colorado, dola, departmen...",Boundaries,Public Domain,DOLA,DOLA - Department of Local Affairs,https://storage.googleapis.com/co-publicdata/A...,DOLA,...,Elizabeth Garner,State Demographer,303-864-7753,Elizabeth.Garner@state.co.us,Todd Bleess,GIS Developer,303-864-7754,todd.bleess@state.co.us,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,City of Denver Playgrounds,This dataset is a polygon representation of pl...,Government,"gocode,bic,recreation",Infrastructure,Open Data Commons Attribution License,City and county of Denver,City of Denver,https://www.denvergov.org/opendata/dataset/cit...,City and county of Denver,...,,,,,,,,,,
444,City of Denver Picnic Areas,This dataset is a point representation of perm...,Government,"recreation,gocode,bic",Infrastructure,Open Data Commons Attribution License,City and county of Denver,City of Denver,https://www.denvergov.org/opendata/dataset/cit...,City and county of Denver,...,,,,,,,,,,
445,City of Denver Trails and Sidewalks,This dataset is a polyline representation of t...,Government,"gocode,bic,recreation,real estate",Infrastructure,Open Data Commons Attribution License,City and county of Denver,City of Denver,https://www.denvergov.org/opendata/dataset/cit...,City and county of Denver,...,,,,,,,,,,
446,City of Denver Golf Courses,This dataset is a point representation of muni...,Government,"gocode,bic,recreation,golf,sporting",Infrastructure,Open Data Commons Attribution License,City and county of Denver,City of Denver,https://www.denvergov.org/opendata/dataset/cit...,City and county of Denver,...,,,,,,,,,,


In [5]:
cim_url_query = 'data.colorado.gov'
cimDatasets={}
with Socrata(cim_url_query, None) as client:
    datasets = client.datasets()
    for dataset in datasets:
        if dataset['owner']['display_name'] == 'Business Intelligence Center of CO':
            title=dataset["resource"]["name"]
            cimDatasets[title]=dataset


In [7]:
cimDatasets['Business Entities in Colorado']

{'resource': {'name': 'Business Entities in Colorado',
  'id': '4ykn-tg5h',
  'resource_name': None,
  'parent_fxf': [],
  'description': 'Recent modifications in the data transformation process may result in data changes.  Additional details about the changes can be found here:</span> https://data.colorado.gov/stories/s/pbek-aaa3\n\nColorado Business Entities (corporations, LLCs, etc.) registered with the Colorado Department of State (CDOS) since 1864.',
  'attribution': 'CDOS',
  'attribution_link': 'https://www.sos.state.co.us/',
  'contact_email': None,
  'type': 'dataset',
  'updatedAt': '2025-10-23T11:16:21.000Z',
  'createdAt': '2014-03-19T22:33:57.000Z',
  'metadata_updated_at': '2025-10-23T11:00:28.000Z',
  'data_updated_at': '2025-10-23T11:16:21.000Z',
  'page_views': {'page_views_last_week': 2807,
   'page_views_last_month': 11138,
   'page_views_total': 205444,
   'page_views_last_week_log': 11.45532722030456,
   'page_views_last_month_log': 13.443332100575896,
   'page_vie

In [9]:
import requests
url = f"https://data.colorado.gov/api/catalog/v1?ids={'farp-k3rw'}"
print("URL ",url)
response = requests.get(url)
datasets = response.json()

URL  https://data.colorado.gov/api/catalog/v1?ids=farp-k3rw


In [10]:
datasets

{'results': [{'resource': {'name': 'Denver Area Golf Courses',
    'id': 'farp-k3rw',
    'resource_name': None,
    'parent_fxf': [],
    'description': 'This dataset is a point representation of municipal golf courses maintained by the Department of Parks and Recreation in the City and County of Denver.',
    'attribution': 'City and county of Denver',
    'attribution_link': 'https://opendata-geospatialdenver.hub.arcgis.com/',
    'contact_email': None,
    'type': 'dataset',
    'updatedAt': '2025-10-23T18:21:34.000Z',
    'createdAt': '2022-06-27T18:02:20.000Z',
    'metadata_updated_at': '2025-10-23T18:21:34.000Z',
    'data_updated_at': '2025-10-20T16:05:46.000Z',
    'page_views': {'page_views_last_week': 18,
     'page_views_last_month': 43,
     'page_views_total': 917,
     'page_views_last_week_log': 4.247927513443585,
     'page_views_last_month_log': 5.459431618637297,
     'page_views_total_log': 9.842350343413809},
    'columns_name': ['the_geom',
     'GOLF_NAME',
    

In [16]:
xenT-cimT

{'Annual Arrests in Colorado by Crime Type for 243 Police Agencies, 1970-2022',
 'Bill Information and Position with Income of Lobbyist in Colorado',
 'CDOT Expenses  ',
 'CDOT Payroll Expenditures  ',
 'CDOT Revenues  ',
 'Census Zip Codes in Colorado 2016',
 'Characterization of Lobbyist Clients in Colorado',
 'Current Surface Water Conditions in Colorado',
 'Development Projects in Grand Junction Colorado 2019',
 'Development Projects in Grand Junction Colorado 2020',
 'Development Projects in Grand Junction Colorado 2021',
 'Development Projects in Grand Junction Colorado 2022',
 'Development Projects in Grand Junction Colorado 2023',
 'Development Projects in Grand Junction Colorado 2024',
 'Directory of Lobbyist Clients in Colorado',
 'Directory of Lobbyists in Colorado',
 'Expenses for Lobbyists in Colorado',
 'Farmers Markets in Colorado 2017',
 'Grocery Stores in Grand Junction Colorado',
 'Highway Mileposts in Colorado',
 'Highway Quality in Colorado 2014 ',
 'Highway Routes 

In [ ]:
cimDatasets.keys()

In [30]:
import requests

# Replace with your Socrata domain and dataset identifier
domain = "data.colorado.gov"
dataset_identifier = "grnx-2zwm"

# Construct the API endpoint URL
#url = f"https://{domain}/resource/{dataset_identifier}.json"

activity_log_endpoint = "/activity_log.json"

# Construct the API endpoint URL  /api/logs.json
#url = f"https://{domain}{activity_log_endpoint}"
url="https://data.colorado.gov/api/activity_log.csv"
# If authentication is required, include your API token
headers = {
    
    "X-App-Token": "JlBIr7bNUkz7s8yDlZcFXFXaN"
}

# Make the API request
response = requests.get(url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    activity_log = response.json()
    # Process the activity log data
    for entry in activity_log:
        print(entry)
else:
    print(f"Error: {response.status_code}")
    print(response.text)

Error: 403
{
  "code" : "authentication_required",
  "error" : true,
  "message" : "You must be logged in to access this resource"
}

